# Session 18 lab: finding structure without labels

You will turn the Lakeside Mart orders into customer segments with RFM analysis and k-means, name each segment, and flag unusual orders and customers.

Rhythm for every step: **PREDICT** first, **run** the cell, **READ** the output, then **CHANGE** one thing. Cells that say `None  # your code here` are yours to fill in. After each answer a `check` line tells you whether your number is right. There is no answer column today, so the checks cover the numbers and your own judgement covers the names. k-means uses random starts, so a few checks accept a range.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, re, zipfile, urllib.request
DATA_LINK = "https://drive.google.com/file/d/11Z5tr2iKuPROn5cyi9P7me6nd3naKfNp/view?usp=sharing"   # the Google Drive link to data.zip that your tutor shares (leave empty to upload data.zip yourself)
if not os.path.exists("orders.csv"):
    if DATA_LINK:
        m = re.search(r"/d/([A-Za-z0-9_-]+)|[?&]id=([A-Za-z0-9_-]+)", DATA_LINK)
        file_id = (m.group(1) or m.group(2)) if m else DATA_LINK
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=" + file_id, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data files ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

---
## Step 1: Keep the loyalty orders

A walk-in order has a blank `customer_id`, so we cannot follow that person over time. Segments are built from loyalty orders only, and we say so in the report.

**PREDICT first.** About 30 percent of orders are walk-ins. How many customers will the loyalty orders cover: fewer than, equal to, or more than the 2,500 rows in customers.csv?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
try:
    import os
    import numpy as np
    import pandas as pd

    os.makedirs("keep", exist_ok=True)   # the folder where you save what you keep
    orders = pd.read_csv("orders.csv", parse_dates=["order_date"])
    loyal = None   # your code here: the orders that have a customer_id (dropna on customer_id)
    print(len(orders), "orders;", len(orders) - len(loyal), "walk-ins left out;", len(loyal), "loyalty orders;", loyal["customer_id"].nunique(), "customers")
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    check("loyalty orders", len(loyal), 21058, 0)
except Exception:
    check('loyalty orders', float("nan"), 0, 0)
try:
    check("customers with a loyalty order", loyal["customer_id"].nunique(), 2499, 0)
except Exception:
    check('customers with a loyalty order', float("nan"), 0, 0)

**READ the output.** Why is it honest to tell the reader that walk-in shoppers are not in the segments?

> My answer: 

---
## Step 2: Build recency, frequency and monetary value (RFM)

One record per customer: days since the last order (recency), number of orders (frequency) and total spend (monetary value). The end date is the last date in the file, so the numbers do not change when you reopen the notebook.

**PREDICT first.** Customer C0001 made 15 orders. Will C0001 have a higher or a lower frequency than the average customer (about 8.4 orders)?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
try:
    end_date = orders["order_date"].max()   # "today" is the last date in the file
    rfm = None   # your code here: group loyal by customer_id; last_order = max of order_date, frequency = count of order_id, spend = sum of order_value_ugx
    rfm["recency"] = None   # your code here: (end_date - rfm["last_order"]).dt.days
    rfm = rfm[["recency", "frequency", "spend"]]
    print(end_date.date())
    rfm.head(3)
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    check("C0001 recency", rfm.loc["C0001", "recency"], 6, 0)
except Exception:
    check('C0001 recency', float("nan"), 0, 0)
try:
    check("C0001 frequency", rfm.loc["C0001", "frequency"], 15, 0)
except Exception:
    check('C0001 frequency', float("nan"), 0, 0)
try:
    check("C0001 spend", rfm.loc["C0001", "spend"], 969690, 0)
except Exception:
    check('C0001 spend', float("nan"), 0, 0)
try:
    check("mean spend", rfm["spend"].mean(), 399687.21, 1)
except Exception:
    check('mean spend', float("nan"), 0, 0)

**READ the output.** The mean spend is above the median spend (print rfm["spend"].median()). What does that tell you about the shape of spend?

> My answer: 

**CHANGE one thing.** Replace end_date by pd.Timestamp.today(). What happens to recency, and why would that be a bad idea in a shared notebook?

> What I saw: 

---
## Step 3: Log transform and standardise

The log transform squeezes the long right tail. Standardisation (z-score) puts the three variables on one scale: mean 0, standard deviation 1. Use ddof=0, the same as scikit-learn's StandardScaler.

**PREDICT first.** Spend has a skewness near 3. After log1p, will its skewness be near 3, near 1 or near 0?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
try:
    logged = None   # your code here: np.log1p of rfm
    scaled = None   # your code here: (logged - logged.mean()) / logged.std(ddof=0)
    print("skewness before:", rfm.skew().round(2).to_dict())
    print("skewness after log:", logged.skew().round(2).to_dict())
    print("scaled standard deviations:", scaled.std(ddof=0).round(3).to_dict())
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    check("skewness of spend before", rfm["spend"].skew(), 3.0856, 0.005)
except Exception:
    check('skewness of spend before', float("nan"), 0, 0)
try:
    check("skewness of spend after log", logged["spend"].skew(), -0.0121, 0.005)
except Exception:
    check('skewness of spend after log', float("nan"), 0, 0)
try:
    check("z-score of C0001 spend", scaled.loc["C0001", "spend"], 1.5656, 0.001)
except Exception:
    check('z-score of C0001 spend', float("nan"), 0, 0)
try:
    check("scaled standard deviation", scaled.std(ddof=0).mean(), 1.0, 0.0001)
except Exception:
    check('scaled standard deviation', float("nan"), 0, 0)

**READ the output.** Which of the three variables was most skewed before the log? Which two numbers on the slide told you that scaling was needed?

> My answer: 

---
## Step 4: k-means with four clusters

StandardScaler does the same z-score step inside scikit-learn. KMeans repeats assign and update until nobody changes cluster. n_init=10 means ten random starts, and the best (lowest inertia) is kept.

**PREDICT first.** The segment numbers 0 to 3 are only labels. Will segment 0 here be the same group as segment 0 on the slides? Roughly how many customers will the largest segment hold?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans

    X = None   # your code here: StandardScaler().fit_transform(logged)
    kmeans = None   # your code here: KMeans with n_clusters=4, n_init=10, random_state=42
    rfm["segment"] = None   # your code here: kmeans.fit_predict(X)
    sizes = np.sort(rfm["segment"].value_counts().values)
    print("inertia:", round(kmeans.inertia_, 1))
    print("segment sizes, smallest to largest:", sizes)
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    check("inertia, k = 4", kmeans.inertia_, 3129.6, 4)
except Exception:
    check('inertia, k = 4', float("nan"), 0, 0)
try:
    check("segment size 1 of 4 (smallest to largest)", sizes[0], 453.0, 40.0)
except Exception:
    check('segment size 1 of 4 (smallest to largest)', float("nan"), 0, 0)
try:
    check("segment size 2 of 4 (smallest to largest)", sizes[1], 560.5, 35.5)
except Exception:
    check('segment size 2 of 4 (smallest to largest)', float("nan"), 0, 0)
try:
    check("segment size 3 of 4 (smallest to largest)", sizes[2], 604.5, 53.5)
except Exception:
    check('segment size 3 of 4 (smallest to largest)', float("nan"), 0, 0)
try:
    check("segment size 4 of 4 (smallest to largest)", sizes[3], 886.0, 36.0)
except Exception:
    check('segment size 4 of 4 (smallest to largest)', float("nan"), 0, 0)

**READ the output.** Which segment is the largest? Do the segment sizes add up to the number of customers?

> My answer: 

**CHANGE one thing.** Optional: add `from sklearn.metrics import silhouette_score` and print `silhouette_score(X, kmeans.labels_)`. A score nearer 1 means tighter, better separated clusters. Compare k = 3, 4 and 5. Remember that k-means assumes round clusters of similar size.

> What I saw: 

**CHANGE one thing.** Change random_state from 42 to 7 and run Step 4 again. Which numbers move, and by how much? What does that say about quoting a segment size to the last customer?

> What I saw: 

---
## Step 5: Choose k with the elbow method

Inertia is the sum of squared distances from each customer to its centroid. It always falls as k grows. We look for the bend.

**PREDICT first.** The inertia for k = 1 is the sum of squares around a single centre. With three standardised variables and 2,499 customers, what number should it be? (Hint: each standardised variable has variance 1.)

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
try:
    inertias = []
    for k in range(1, 9):
        model = None   # your code here: KMeans with n_clusters=k, n_init=10, random_state=42, fitted on X
        inertias.append(None)   # your code here: the model's inertia_
    print([round(v) for v in inertias])

    import matplotlib.pyplot as plt
    plt.plot(range(1, 9), inertias, marker="o")
    plt.xlabel("k: number of clusters")
    plt.ylabel("inertia")
    plt.show()
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    check("inertia, k = 1", inertias[0], 7497.0, 0.5)
except Exception:
    check('inertia, k = 1', float("nan"), 0, 0)
try:
    check("inertia, k = 4", inertias[3], 3129.6, 5)
except Exception:
    check('inertia, k = 4', float("nan"), 0, 0)
try:
    check("inertia, k = 8", inertias[7], 2045.7, 40)
except Exception:
    check('inertia, k = 8', float("nan"), 0, 0)
try:
    check("inertia always falls", bool(np.all(np.diff(inertias) < 0)), 1, 0)
except Exception:
    check('inertia always falls', float("nan"), 0, 0)

**READ the output.** Where does the line bend? Is the bend sharp or gentle? Would you still choose k = 4, and what is your business reason?

> My answer: 

---
## Step 6: PCA: a flat picture of three variables

Principal component analysis (PCA) is a dimensionality reduction method. Each principal component is a weighted mix of the three variables, ordered by how much variance it keeps.

**PREDICT first.** If recency, frequency and spend were completely unrelated, how much variance would each component keep? Here they are related, so will PC1 keep more or less than a third?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
try:
    from sklearn.decomposition import PCA

    pca = None   # your code here: PCA with n_components=3
    components = None   # your code here: pca.fit_transform(X)
    ratio = None   # your code here: pca.explained_variance_ratio_
    print("explained variance:", ratio.round(3), "first two:", round(ratio[:2].sum(), 3))

    plt.scatter(components[:, 0], components[:, 1], c=rfm["segment"], s=6, cmap="viridis")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.show()
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    check("PC1 explained variance", ratio[0], 0.5736, 0.001)
except Exception:
    check('PC1 explained variance', float("nan"), 0, 0)
try:
    check("PC2 explained variance", ratio[1], 0.2911, 0.001)
except Exception:
    check('PC2 explained variance', float("nan"), 0, 0)
try:
    check("PC3 explained variance", ratio[2], 0.1353, 0.001)
except Exception:
    check('PC3 explained variance', float("nan"), 0, 0)

**READ the output.** How much of the variance do two components keep? Do the colours (the k-means segments) separate in the picture?

> My answer: 

---
## Step 7: Profile, name and act

A segment is only useful if it has a name a manager understands and an action. Read the averages in real units, then choose the names. Your names may differ from the slides.

**PREDICT first.** Which segment will hold the most spend per customer? Will it also have the lowest recency?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
try:
    profile = None   # your code here: group rfm by segment; customers = size, recency = mean of recency, orders = mean of frequency, spend = mean of spend; round to 0 places
    profile["share_of_spend_pct"] = (rfm.groupby("segment")["spend"].sum() / rfm["spend"].sum() * 100).round(0)
    print(profile)
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    segment_names = None   # your code here: a dict that maps each segment number in profile to a name in business words, for example {0: "Best customers", ...}
    rfm["segment_name"] = rfm["segment"].map(segment_names)
    rfm.reset_index().to_csv("keep/customer_segments.csv", index=False)   # your keepsake
    print(rfm["segment_name"].value_counts())
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    check("four segments", len(profile), 4, 0)
except Exception:
    check('four segments', float("nan"), 0, 0)
try:
    check("customers add up", profile["customers"].sum(), 2499, 0)
except Exception:
    check('customers add up', float("nan"), 0, 0)
try:
    check("best segment: average spend", profile["spend"].max(), 833080, 70000)
except Exception:
    check('best segment: average spend', float("nan"), 0, 0)
try:
    check("best segment: average recency in days", profile.loc[profile["spend"].idxmax(), "recency"], 39.7, 12)
except Exception:
    check('best segment: average recency in days', float("nan"), 0, 0)
try:
    check("four different names", len(set(segment_names.values())), 4, 0)
except Exception:
    check('four different names', float("nan"), 0, 0)

**READ the output.** Write one action for each segment in the cell below. Which action would cost the least to test?

> My answer: 

## Your segment actions

Double click and fill in the table.

| Segment name | Who they are | What I would do |
|---|---|---|
| | | |
| | | |
| | | |
| | | |

---
## Step 8: Flag unusual orders and customers

Rule one is the percentile rule on order value (all orders, walk-ins included). Rule two is an IsolationForest on the scaled RFM variables, which isolates odd customers with few random cuts.

**PREDICT first.** The percentile rule flags the top 1 percent of 30,229 orders. How many orders is that? Will the median of the flagged orders be 2, 10 or 35 times the median of all orders?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
try:
    p99 = None   # your code here: the 99th percentile of order_value_ugx in orders (quantile(0.99))
    big = None   # your code here: the orders with order_value_ugx above p99
    print("99th percentile:", round(p99), "UGX;", len(big), "orders flagged;", big["customer_id"].isna().sum(), "of them walk-ins")
    print("median of flagged:", big["order_value_ugx"].median(), "median of all:", orders["order_value_ugx"].median())

    from sklearn.ensemble import IsolationForest

    iso_model = None   # your code here: IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
    rfm["anomaly"] = None   # your code here: iso_model.fit_predict(X) == -1 (the value -1 means flagged)
    flagged = rfm[rfm["anomaly"]]
    print(len(flagged), "customers flagged")
    print(flagged[["recency", "frequency", "spend"]].median())
    big.to_csv("keep/unusual_orders.csv", index=False)   # your list to review
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    check("99th percentile of order value", p99, 468620.4, 1)
except Exception:
    check('99th percentile of order value', float("nan"), 0, 0)
try:
    check("orders above it", len(big), 303, 0)
except Exception:
    check('orders above it', float("nan"), 0, 0)
try:
    check("customers flagged (1 percent of 2,499)", len(flagged), 25, 2)
except Exception:
    check('customers flagged (1 percent of 2,499)', float("nan"), 0, 0)
try:
    unusual = ((flagged["recency"] > 100) & (flagged["spend"] < 100000)) | (flagged["spend"] > 800000)
except Exception:
    pass
try:
    check("most flagged customers are lapsed low spenders or heavy spenders", unusual.mean() >= 0.5, 1, 0)
except Exception:
    check('most flagged customers are lapsed low spenders or heavy spenders', float("nan"), 0, 0)

In [ ]:
show_checks()   # run this to see how many checks pass

**READ the output.** Look at the flagged customers. Which are the lapsed low spenders and which are heavy spenders? Name one thing you would check before treating any of them as an error.

> My answer: 

**CHANGE one thing.** Change contamination to 0.05. How many customers are flagged now? What does that tell you about the rule?

> What I saw: 

## Keep

Save the notebook. Keep `keep/customer_segments.csv` (every loyalty customer with RFM, a segment name) and `keep/unusual_orders.csv` (orders to review). Bring one question about forecasting to Session 19.